## Evaluation for the non-reasoning benchmark

### Using a Binary Scoring System from Evidently.ai 

In [5]:
import pandas as pd
import numpy as np

from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently import BinaryClassification
from evidently.descriptors import *
from evidently.presets import TextEvals, ValueStats, ClassificationPreset
from evidently.metrics import *

from evidently.llm.templates import BinaryClassificationPromptTemplate
from evidently.llm.templates import BinaryClassificationPromptTemplate, MulticlassClassificationPromptTemplate 
from evidently.descriptors import LLMEval, ToxicityLLMEval, ContextQualityLLMEval, DeclineLLMEval, CorrectnessLLMEval

/home/baslrou/.local/lib/python3.9/site-packages/evidently/core/metric_types.py:376: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  np_bool = np.bool  # type: ignore[attr-defined]


In [6]:
# Load your CSV file (replace with your filename)
df= pd.read_csv("gpt-4o_eval_with_bert_REFERENCE - full.csv")
# df.head(10)

In [3]:
df_full = df[df['question type'] == 'Reasoning Questions']
# df_full = df_full[df_full['Evaluation'].isin(['0', '1'])]

df_full.Evaluation.info


<bound method Series.info of 75       1
76       1
77       1
78       0
79       1
      ... 
149    NaN
150    NaN
151    NaN
152    NaN
153    NaN
Name: Evaluation, Length: 79, dtype: object>

In [4]:
df_full.head(2)

,Question,Expected Answer,Model Answer,Prompt,Evaluation,BERT Score,Unnamed: 6,question type
75,Is there a specific PC and address combination...,Ans: None exists,"Based on the data provided, we need to identif...",You are an expert in computer architecture and...,1,0.781804,NaN,Reasoning Questions
76,Identify a PC in the astar workload where MLP ...,Ans: None exists,To identify a point of comparison where the ML...,You are an expert in computer architecture and...,1,0.767162,NaN,Reasoning Questions


### Multi Column Analysis 1
#### Scoring the alignment of answer key with the LLM response

In [39]:
# relevance = MulticlassClassificationPromptTemplate(   
#         pre_messages=[("system", "You are an expert evaluator for the alignment between resposnses and answer-keys.")],   
#         criteria = """ You are given a response. Assess its alignment with the answer key.
#         Answer-Key:
#         {Expected Answer}
#         """,
#         additional_columns={"Expected Answer": "Expected Answer"},
#         category_criteria = {
#             "incorrect" : "The answer is contradictory to the answer key",
#             "partially correct" : "The answer is neutral or general and only somewhat relates to the answer key.",
#             "correct": "The answer fully aligns with the answer key",
#         },
#         uncertainty="unknown",
#         include_reasoning=True,
#         include_score=True,
#         )

### Multi Column Analysis 2
#### Scoring the response based on a reasoning criteria 

In [10]:
relevance = MulticlassClassificationPromptTemplate(   
        pre_messages=[("system", "You are an expert in Computer Architecture and will evaluate the quality of answers to the questions")],   
        criteria = """ You are given a response. Assess its quality.
        question:
        {Prompt}
        """,
        additional_columns={"Prompt": "Prompt"},
        category_criteria = {
            "Bad" : "The answer is an incorrect threat of thoughts or logic",
            "neutral" : "The answer is neutral or general and only somewhat explains appropriate answers.",
            "Somewhat Good": "The answer follows a logical and meaningful discussion but does not answer the question correclty conclusively",
            "Good": "The answer follows a logical and meaningful discussion and answers the question correclty",
        },
        uncertainty="unknown",
        include_reasoning=True,
        include_score=True,
        )

In [11]:
llm_evals = Dataset.from_pandas(
    pd.DataFrame(df_full),
    data_definition=DataDefinition(),
    descriptors=[
        LLMEval("Model Answer", 
                template=relevance, 
                additional_columns={"Prompt": "Prompt"},
                provider = "openai", 
                model = "gpt-4o-mini", 
                alias="Evaluation")],
    )

In [12]:
llm_evals.as_dataframe()

,Question,Expected Answer,Model Answer,Prompt,Evaluation,BERT Score,Unnamed: 6,question type,Evaluation_1,Evaluation score_Bad,Evaluation score_neutral,Evaluation score_Somewhat Good,Evaluation score_Good,Evaluation reasoning
75,Is there a specific PC and address combination...,Ans: None exists,"Based on the data provided, we need to identif...",You are an expert in computer architecture and...,1,0.781804,NaN,Reasoning Questions,Somewhat Good,0.0,0.20,0.6,0.20,The response provides a logical analysis of th...
76,Identify a PC in the astar workload where MLP ...,Ans: None exists,To identify a point of comparison where the ML...,You are an expert in computer architecture and...,1,0.767162,NaN,Reasoning Questions,Somewhat Good,0.0,0.30,0.6,0.10,The response provided a logical analysis of th...
77,Which cache replacement policy achieves the hi...,Ans: Belady. Miss Rates: LRU = 68.77%; MLP = 8...,To determine which cache replacement policy ac...,You are an expert in computer architecture and...,1,0.801766,NaN,Reasoning Questions,Good,0.0,0.00,0.0,1.00,The response accurately analyzes the cache per...
78,What is the number of incorrect evictions for ...,Ans: 26200.,80.06%.,You are an expert in computer architecture and...,0,0.844454,XXXXXXX,Reasoning Questions,Good,0.0,0.00,0.0,1.00,The provided answer of '80.06%' correctly stat...
79,Analyze how cache line scores vary across evic...,Ans:,To analyze the cache line scores for memory ad...,You are an expert in computer architecture and...,1,0.799740,NaN,Reasoning Questions,Good,0.0,0.10,0.3,0.60,The response provides a thorough analysis of h...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,On which workload does the LRU policy have the...,Ans:,The LRU policy has the most incorrect eviction...,You are an expert in computer architecture and...,NaN,0.829807,NaN,Reasoning Questions,Good,0.0,0.00,0.2,0.80,The response correctly identifies that the LRU...
150,On which workload does the PARROT policy have ...,Ans:,To determine the workload on which the PARROT ...,You are an expert in computer architecture and...,NaN,0.805456,NaN,Reasoning Questions,Good,0.0,0.00,0.0,1.00,The answer correctly identifies that the PARRO...
151,For which workload did all of the policies str...,Ans:,The workload under which all the policies stru...,You are an expert in computer architecture and...,NaN,0.814313,NaN,Reasoning Questions,Good,0.0,0.00,0.2,0.80,The answer correctly identifies the 'mcf' work...
152,On which workload does the MLP policy have the...,Ans:,To determine the workload where the MLP policy...,You are an expert in computer architecture and...,NaN,0.798207,NaN,Reasoning Questions,Good,0.0,0.00,0.1,0.90,The response correctly identifies the 'mcf' wo...
